# 第 3 章 光学扫描全息原理 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/scanholo/ch3/`。

## background · 背景相除

`background_division.py`

In [ ]:
# ===== 拓展实战（编者，原书无）· 同轴数字全息的背景相除法 =====
# 改编自你已有的 holography.m / 孪生像.py / 背景相除法.py（规范化重写，去 OpenCV 依赖）：
# 过冷水滴（振幅 0.95 + 相位 1.2 rad 的圆盘）经 z=5cm 角谱传播生成同轴全息图，
# 相机面叠加高斯激光背景与两粒灰尘 -> 背景相除 I_raw/I_bg -> sqrt 估计复振幅 -> 反向传播重建。
#
# 与本章 OSH 的关系（笔记 3.5 节"孪生像问题"对照）：
# - 背景相除只能去掉乘性背景（灰尘/照明不均），对同轴全息的孪生像虚影无能为力——
#   重建振幅图里水滴周围仍有发散的"孪生像波纹"。
# - OSH 路线（本章）：一次 2-D 扫描同时记录 sin/cos 两张全息图，复数相加 H_cos + j*H_sin
#   即可在同轴记录下彻底消除孪生像（式 3.5-5）。
#
# 严格书约定：前向变换 ifft2*N^2（+j 指数）、逆变换 fft2/N^2（-j 指数）——见 ch2 asm_propagation.py。
#
# MATLAB / Python 差异提示：
# 1) MATLAB 版用 regionprops 做连通域定位；此处用简单阈值 + 质心公式（numpy 足够）。
# 2) imagesc/imshow 坐标原点差异同前几章。

import numpy as np
import matplotlib.pyplot as plt

# ----- 1. 参数与物体/背景 -----
N = 512
lam = 532e-9
dx = 5e-6
L = N*dx
z_prop = 0.05
k0 = 2*np.pi/lam
x = (np.arange(N)-N//2)*dx
y = (np.arange(N)-N//2)*dx
X, Y = np.meshgrid(x, y)

w0 = 1.5e-3                                        # 高斯激光背景束腰
I_laser = np.exp(-2*(X**2+Y**2)/w0**2)
t_noise = np.ones((N, N))
dust1 = ((X-0.5e-3)**2 + (Y-0.5e-3)**2) <= (20e-6)**2
dust2 = ((X+0.6e-3)**2 + (Y-0.4e-3)**2) <= (15e-6)**2
t_noise[dust1 | dust2] = 0.2                       # 两粒灰尘（透过率 0.2）
I_bg = I_laser*t_noise

U_obj = np.ones((N, N), dtype=complex)
mask = (X**2 + Y**2) <= (50e-6)**2                 # 半径 50 um 的过冷水滴
U_obj[mask] = 0.95*np.exp(1j*1.2)                  # 振幅 0.95、相位 1.2 rad

# ----- 2. 书式傅里叶协议 + 角谱正/反向传播 -----
def ft2_book(U):
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(U))) * N**2

def ift2_book(A):
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(A))) / N**2

df = 1/L
fx = (np.arange(N)-N//2)*df
FY, FX = np.meshgrid(fx, fx)
KX, KY = 2*np.pi*FX, 2*np.pi*FY
root = 1 - (KX**2+KY**2)/k0**2
root[root < 0] = 0                                 # 滤除消逝波
H_fwd = np.exp(-1j*k0*z_prop*np.sqrt(root))
H_bwd = np.exp(+1j*k0*z_prop*np.sqrt(root))        # 反向传播：z 取 -z_prop

U_det = ift2_book(ft2_book(U_obj)*H_fwd)           # 传播到相机面
I_raw = np.abs(U_det)**2 * I_bg                    # 相机记录（含背景）
I_proc = I_raw/I_bg                                # 背景相除（乘性噪声消除）
U_rec = ift2_book(ft2_book(np.sqrt(I_proc))*H_bwd) # 反向传播重建（sqrt 强度当复振幅估计）

# ----- 3. 简单定位（阈值 + 质心，代替 OpenCV 连通域） -----
amp = np.abs(U_rec)
amp_n = (amp - amp.min())/(amp.max()-amp.min())
binary = amp_n < 0.45
ys, xs = np.nonzero(binary)
if len(xs) > 0:
    cx, cy = xs.mean(), ys.mean()
    d_equiv = 2*dx*np.sqrt(binary.sum()/np.pi)*1e6
    print(f'水滴质心：x = {(cx-N/2)*dx*1e3:+.3f} mm, y = {(cy-N/2)*dx*1e3:+.3f} mm, '
          f'等效直径 = {d_equiv:.1f} um（真值 100 um）')
    print('注意：重建振幅中水滴周围仍可见孪生像虚影波纹——背景相除无法消除孪生像。')

# ----- 4. 可视化 -----
fig, ax = plt.subplots(2, 3, figsize=(14, 8))
ext = [x[0]*1e3, x[-1]*1e3, y[0]*1e3, y[-1]*1e3]
ax[0,0].imshow(np.abs(U_obj), extent=ext, cmap='gray')
ax[0,0].set_title('1. Object amplitude |U_obj| (z=0)')
ax[0,1].imshow(I_raw, extent=ext, cmap='gray')
ax[0,1].set_title('2. Raw hologram (laser bg + dust)')
ax[0,2].imshow(I_proc, extent=ext, cmap='gray')
ax[0,2].set_title('3. Background-divided hologram')
ax[1,0].imshow(amp, extent=ext, cmap='gray')
ax[1,0].set_title('4. Reconstructed amplitude (twin-image haze)')
ax[1,1].imshow(np.angle(U_rec), extent=ext, cmap='jet')
ax[1,1].set_title('5. Reconstructed phase (rad)')
ax[1,2].imshow(amp, extent=ext, cmap='gray')
if len(xs) > 0:
    ax[1,2].plot((cx-N/2)*dx*1e3, (cy-N/2)*dx*1e3, 'ro', ms=10, mfc='none', mew=2)
    ax[1,2].set_title('6. Centroid detection')
for a in ax.flat:
    a.set_xlabel('x (mm)'); a.set_ylabel('y (mm)')
plt.tight_layout(); plt.show()


## heterodyne · detection

`heterodyne_detection.py`

In [ ]:
# ===== 书 3.2 节 · 光学外差 + 电子多路检测（式 3.2-1 ~ 3.2-4） =====
# 时域仿真：信息光 psi_p = A exp(j*phi) exp(j*w0 t) 与参考光 B exp[j(w0+Omega)t]
# 在光电探测器上拍频，电流 (3.2-1)：
#     i(t) = D [ A^2 + B^2 + 2AB cos(Omega t - phi) ]
# 电子学链路（图 3.6）：BPF@Omega 提取外差电流 -> 上通道乘 cos(Omega t) + LPF 得同相分量 i_c (3.2-3a)
# -> 下通道乘 sin(Omega t) + LPF 得正交分量 i_s (3.2-3b) -> 复数加法 i_c + j*i_s = A exp(j*phi) (3.2-4)
#
# 验证结论（本笔记实跑）：A_rec = 0.700000（误差 2e-16）、phi_rec = 1.300000 rad（误差 2e-15）。
# 即外差电流完整保留信息光的幅度与相位；lock-in 解调可无损恢复。
#
# MATLAB / Python 差异提示：
# 1) MATLAB 的 FFT 前向核是 e^{-j...}，numpy 同为 e^{-j...}，此处只做频域选带，无约定问题。
# 2) 理想带通/低通直接用 FFT 选带与时间平均实现；实际系统用模拟滤波器，会引入纹波与延迟。

import numpy as np
import matplotlib.pyplot as plt

# ----- 1. 参数 -----
A, phi, B = 0.7, 1.3, 1.0        # 信息光幅度/相位（真值）、参考光幅度（实数）
Omega = 2*np.pi*1.0e6            # 外差（拍频）频率 1 MHz
fs = 64e6                        # 采样率
T = 8e-6                         # 总时长 8 us（8 个拍频周期）
t = np.arange(0, T, 1/fs)

# ----- 2. 光电探测电流（式 3.2-1，取探测器面积 D=1） -----
i_t = A**2 + B**2 + 2*A*B*np.cos(Omega*t - phi)      # DC(基带) + AC(外差)

# ----- 3. BPF @ Omega：提取外差电流 i_Omega ∝ 2AB cos(Omega t - phi) -----
I_f = np.fft.rfft(i_t)
f_axis = np.fft.rfftfreq(len(t), 1/fs)
band = (f_axis > 0.5e6) & (f_axis < 1.5e6)           # 理想带通（保留 Omega，滤掉直流基带）
iOmega = np.fft.irfft(np.where(band, I_f, 0), len(t))

# ----- 4. lock-in 解调（式 3.2-2 / 3.2-3）：乘 cos/sin 后低通（时间平均） -----
seg = slice(int(0.5*len(t)), None)                   # 丢弃滤波器建立段
ic = 2*np.mean(iOmega[seg]*np.cos(Omega*t[seg]))     # i_c = A cos(phi)   (3.2-3a)
is_ = 2*np.mean(iOmega[seg]*np.sin(Omega*t[seg]))    # i_s = A sin(phi)   (3.2-3b)
A_rec = np.hypot(ic, is_)/2                          # 幅度（含常数因子 2，归一化后）
phi_rec = np.arctan2(is_, ic)                        # 相位
print(f'恢复幅度 A = {A_rec:.6f}（真值 {A}），误差 {abs(A_rec-A):.1e}')
print(f'恢复相位 phi = {phi_rec:.6f} rad（真值 {phi}），误差 {abs(phi_rec-phi):.1e}')
print('式(3.2-4)：i_c + j*i_s ∝ A exp(j*phi) —— 幅度与相位完整恢复')

# ----- 5. 可视化（对应笔记图 sim-heterodyne.png） -----
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0,0].plot(t*1e6, i_t, lw=0.8)
ax[0,0].set_title('i(t) = D[A$^2$+B$^2$+2AB cos($\\Omega$t-$\\phi$)]   (Eq. 3.2-1)')
ax[0,0].set_xlabel('t (us)'); ax[0,0].set_ylabel('i(t)')
ax[0,1].plot(f_axis/1e6, np.abs(I_f), lw=0.8); ax[0,1].set_xlim(0, 4)
ax[0,1].set_title('spectrum: DC(baseband) + heterodyne line at $\\Omega$')
ax[0,1].set_xlabel('f (MHz)')
ax[1,0].plot(t*1e6, iOmega, lw=0.8)
ax[1,0].set_title('after BPF @ $\\Omega$:  i$_\\Omega$ = 2AB cos($\\Omega$t-$\\phi$)')
ax[1,0].set_xlabel('t (us)')
ax[1,1].plot(t*1e6, iOmega*np.cos(Omega*t), lw=0.6, label='i$_\\Omega\\times$cos$\\Omega$t')
ax[1,1].plot(t*1e6, iOmega*np.sin(Omega*t), lw=0.6, alpha=0.6, label='i$_\\Omega\\times$syn$\\Omega$t')
ax[1,1].axhline(ic/2, color='r', ls='--', label=f'LPF -> i_c/2 = {ic/2:.3f}')
ax[1,1].axhline(-is_/2, color='b', ls=':', label=f'LPF -> -i_s/2 = {-is_/2:.3f}')
ax[1,1].set_title('lock-in detection (Eq. 3.2-2 / 3.2-3)')
ax[1,1].set_xlabel('t (us)'); ax[1,1].legend(fontsize=8)
plt.tight_layout(); plt.show()


## osh · simulation

`osh_simulation.py`

In [ ]:
# ===== 书 3.5 节 Example 3.1 · OSH.m 的规范化重写（Table 3.1，复现原书图 3.15） =====
# 原书 OSH.m（改编自 Poon & Banerjee《Contemporary Optical Image Processing with MATLAB》Table 7.2）：
#   读入 256x256 图像 I(x,y)（原书 vatech.bmp）-> 频域乘 OTFosh -> ifft2 得复全息图 H
#   real(H) 显示为 sine-FZP hologram (3.5-8a)、imag(H) 显示为 cosine-FZP hologram (3.5-8b)
#   重建：fft2(H_any).*conj(OTFosh) 后 ifft2（式 3.5-10），sin/cos 单独重建有孪生噪声，
#         复全息图（real + j*imag）重建干净；H_c-（real - j*imag）重建离焦（图 3.15g）。
#
# 约定说明（重要）：
# 原书 OSH.m 用 MATLAB 原生 fft2（-j 指数）当"前向"，OTFosh 相应取 exp(-j*sigma*k^2)（-j 版），
# 与书正文公式（+j 约定、OTF = exp[+j...] 的共轭）互为镜像——但两者各自自洽，闭环重建结果相同。
# 本脚本统一到"书式协议"（前向 +j = ifft2*N^2），OTFosh = exp(-j*sigma*k^2) 即书 (3.5-1a) 本身：
#   H = ift2_book{ ft2_book{I} * OTFosh }  =>  F{H} = F{I}*OTFosh
#   重建 = ift2_book{ ft2_book{H} * conj(OTFosh) } = ift2_book{F{I}} = I（|OTF|=1，相位精确抵消）
#   实测：复全息图重建 vs 原图相关系数 = 1.0000。
# 电子学符号：real(H) = H_sin（sine 全息图）、imag(H) = -H_cos（显示时翻转灰度，无碍）。
#
# MATLAB / Python 差异提示：
# 1) 原书 OSH.m 的 FI=fftshift(fft2(I)) 中心化后直接 ifft2(FH)（未做 ifftshift 对齐），会带入
#    (-1)^(i+j) 棋盘相位——强度显示无碍，规范写法应 ifftshift 后再逆变换（本版已修正）。
# 2) 原书 sigma = z0/2k0 = 2.0（任意单位）、kx,ky = -12.8:0.1:12.8。此网格下相邻格相位跳变
#    2*sigma*k*dk 最大约 5.1 rad > pi，属于欠采样演示（书图 3.15b/c 颗粒状外观的来源之一）；
#    但闭环重建 self-conjugate 抵消，结论不受影响。物理单位的严谨版本见 point_target_osh.py。
# 3) MATLAB 1-based/Python 0-based；plt.imshow 加 origin='lower' 对齐数学坐标。

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

# ----- 1. 输入图像（原书 vatech.bmp；此处用 PIL 生成 "VT / OSH" 文字图代替） -----
N = 256
img = Image.new('L', (N, N), 0)
d = ImageDraw.Draw(img)
try:
    fnt = ImageFont.truetype('arial.ttf', 88)
except Exception:
    fnt = ImageFont.load_default()
d.text((128, 80), 'VT', font=fnt, fill=255, anchor='mm')
d.text((128, 185), 'OSH', font=fnt, fill=255, anchor='mm')
I = (np.asarray(img) > 80).astype(float)

# ----- 2. 书式傅里叶协议 + OSH 传递函数（sigma = z0/2k0 = 2.0，任意单位，同原书） -----
def ft2_book(U):
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(U))) * N**2

def ift2_book(A):
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(A))) / N**2

sigma = 2.0
ku = np.arange(N)*0.1 - 12.8
KY, KX = np.meshgrid(ku, ku)
OTFosh = np.exp(-1j*sigma*(KX**2+KY**2))           # 式 (3.5-1a)

# ----- 3. 记录（Coding）：频域乘 OTFosh -> 逆变换得复全息图 -----
H = ift2_book(ft2_book(I)*OTFosh)
Hsin = np.real(H)                                  # sine-FZP hologram   (3.5-8a)
HcosM = np.imag(H)                                 # imag 支路（= -H_cos，显示为 cosine  (3.5-8b)）

# ----- 4. 解码（Decoding，式 3.5-10：频域乘 conj(OTFosh) 模拟与 h(x,y;z0) 卷积） -----
rec_sine = ift2_book(ft2_book(Hsin)*np.conj(OTFosh))      # 孪生噪声
rec_cosine = ift2_book(ft2_book(HcosM)*np.conj(OTFosh))   # 孪生噪声
Hc_plus = H                                    # complex = real + j*imag（书 OSH.m 惯例，= i_Omegap）
Hc_minus = Hsin - 1j*HcosM                     # H_c- 型（实像共轭 -> 虚像/离焦）
rec_cplus = ift2_book(ft2_book(Hc_plus)*np.conj(OTFosh))    # 干净
rec_cminus = ift2_book(ft2_book(Hc_minus)*np.conj(OTFosh))  # 离焦（图 3.15g）

def cc(a, b):
    a = a - a.mean(); b = b - b.mean()
    return abs(np.vdot(a, b))/np.sqrt(np.vdot(a, a).real*np.vdot(b, b).real)

print(f'复全息图 (real+j*imag) 重建 vs 原图：相关系数 = {cc(I, rec_cplus):.4f}')
print(f'H_c- (real-j*imag) 重建 vs 原图：相关系数 = {cc(I, rec_cminus):.4f}（严重离焦）')
rs = np.abs(rec_sine)**2
print(f'sin 全息图单独重建：能量集中度 = {rs.max()/rs.sum():.4f}（孪生像使能量弥散）')

# ----- 5. 可视化（对应原书图 3.15 a)~g)） -----
fig, ax = plt.subplots(2, 4, figsize=(15, 7.2))
ax[0,0].imshow(I, cmap='gray'); ax[0,0].set_title('a) Original image I(x,y)')
ax[0,1].imshow(Hsin, cmap='gray'); ax[0,1].set_title('b) Sine-FZP hologram (3.5-8a)')
ax[0,2].imshow(HcosM, cmap='gray'); ax[0,2].set_title('c) Cosine-FZP hologram (3.5-8b)')
ax[0,3].imshow(np.abs(H), cmap='gray'); ax[0,3].set_title('|complex hologram|')
ax[1,0].imshow(np.abs(rec_sine), cmap='gray', vmax=np.abs(rec_sine).max()*0.5)
ax[1,0].set_title('d) Rec. sine-holo (twin noise)')
ax[1,1].imshow(np.abs(rec_cosine), cmap='gray', vmax=np.abs(rec_cosine).max()*0.5)
ax[1,1].set_title('e) Rec. cosine-holo (twin noise)')
ax[1,2].imshow(np.abs(rec_cplus), cmap='gray')
ax[1,2].set_title('f) Rec. complex H_c+ (clean)')
ax[1,3].imshow(np.abs(rec_cminus), cmap='gray')
ax[1,3].set_title('g) Rec. H_c- (defocused, fig 3.15g)')
for a in ax.flat:
    a.axis('off')
plt.tight_layout(); plt.show()


## point · target · osh

`point_target_osh.py`

In [ ]:
# ===== 书 3.5 节 · 点目标扫描全息：sin/cos FZP 全息图与复全息图重建（式 3.5-1 ~ 3.5-10） =====
# 物理设置：p1(x,y)=1（光瞳全开 -> L1 后焦面上点源，频率 w0）、p2(x,y)=delta（-> 平面波，频率 w0+Omega）。
# 点目标 |Gamma_0|^2 = delta(x,y)delta(z-z0) 位于距扫描镜 z0 处。
# 主结果：
#   H_sin(x,y) = (k0/2*pi*z0) sin[(k0/2*z0)(x^2+y^2)]           (3.5-4a)
#   H_cos(x,y) = (k0/2*pi*z0) cos[(k0/2*z0)(x^2+y^2)]           (3.5-4b)
#   H_c+ = H_cos + j*H_sin = j*i_Omegap  ->  与 h(x,y;z0) 卷积聚焦成点（conc = 1.0000）
#   H_c- = H_cos - j*H_sin = -j*conj(i_Omegap) -> 同样卷积得到 2*z0 处菲涅耳衍射图（corr = 1.0000）
#   sin 全息图单独重建：主像只占一半能量（另一半给孪生像/离焦斑，实测 0.501）——孪生像问题。
#
# 约定（全书）：前向变换 +j 指数 -> 用 ifft2*N^2；逆变换 -j 指数 -> 用 fft2/N^2。
# OTF_osh(kx,ky;z) = exp[-j*z/(2*k0)*(kx^2+ky^2)]  (3.5-1a)，|OTF|=1（纯相位）。
# 电子学符号：i_c = Re{i_Omegap}（cos 解调）、i_s = -Im{i_Omegap}（sin 解调，注意负号，
#             与式 (3.2-3b) 一致；原书 (3.4-14b) 直接写 Im，见笔记勘误）。
#
# MATLAB / Python 差异提示：
# 1) MATLAB 1-based、Python 0-based：中心像素 MATLAB (257,257) <-> Python (256,256)。
# 2) imshow 需 origin='lower' 才与数学坐标一致；MATLAB imagesc 配 axis image。
# 3) e^{-j sigma k^2} 不是带限函数：离散记录的 sin/cos 全息图含 Poisson 旁瓣混叠
#    （书 fig 3.15b/c 颗粒状外观的数值根源），但闭环重建用 conj(OTFosh) 精确抵消，不受影响。

import numpy as np
import matplotlib.pyplot as plt

# ----- 1. 物理参数（真实单位） -----
lam = 0.6328e-6            # He-Ne 红光波长（m）
k0 = 2*np.pi/lam           # 波数
z0 = 0.30                  # 点目标到扫描镜的距离（m）
N = 512                    # 网格
dx = 10e-6                 # 像素尺寸（m）
sigma = z0/(2*k0)          # = z0/(2*k0)，式 (3.5-1a) 的二次相位系数

x = (np.arange(N)-N//2)*dx
XX, YY = np.meshgrid(x, x)
df = 1/(N*dx)                                  # DFT 频率格距
kx = (np.arange(N)-N//2)*df
KY, KX = np.meshgrid(kx, kx)

# ----- 2. 书式傅里叶协议（前向 +j = ifft2*N^2，逆向 -j = fft2/N^2） -----
def ft2_book(U):
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(U))) * N**2

def ift2_book(A):
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(A))) / N**2

# ----- 3. OSH 传递函数 (3.5-1a) 与点目标记录 -----
OTFosh = np.exp(-1j*sigma*(KX**2+KY**2))
I_pt = np.zeros((N, N)); I_pt[N//2, N//2] = 1.0
iOmegap = ift2_book(ft2_book(I_pt)*OTFosh)     # i_Omegap = F^{-1}{F{|Gamma0|^2} OTF_osh}  (3.4-15)

Hsin = np.real(iOmegap)                        # H_sin = Re{i_Omegap}   (3.5-3a)
Hcos = -np.imag(iOmegap)                       # H_cos = -Im{i_Omegap}  (3.5-3b，电子学负号)
Hc_plus  = Hcos + 1j*Hsin                      # H_c+ = H_cos + j*H_sin = j*i_Omegap    (3.5-5)
Hc_minus = Hcos - 1j*Hsin                      # H_c- = H_cos - j*H_sin = -j*conj(i_Omegap)

# ----- 4. 数字重建（式 3.5-10：与 h(x,y;z0) 卷积 = 频域乘 OTF_osh*） -----
rec_cp = ift2_book(ft2_book(Hc_plus)*np.conj(OTFosh))    # 聚焦 -> 点（实像）
rec_cm = ift2_book(ft2_book(Hc_minus)*np.conj(OTFosh))   # 离焦 -> 2*z0 菲涅耳衍射
rec_sp = ift2_book(ft2_book(Hsin)*np.conj(OTFosh))       # sin 全息图单独 -> 点 + 孪生

rcp = np.abs(rec_cp)**2
pk = np.unravel_index(np.argmax(rcp), rcp.shape)
conc = rcp[pk]/rcp.sum()
rsm = np.abs(rec_sp)**2
ratio_sin = rsm[pk]/rsm.sum()

# ----- 5. 对照：H_c- 重建 vs 原物在 2*z0 处的菲涅耳衍射（书 p.88 文字结论） -----
H2 = np.exp(-1j*k0*2*z0)*np.exp(1j*(KX**2+KY**2)*(2*z0)/(2*k0))
fres2 = ift2_book(ft2_book(I_pt)*H2)

def cc(a, b):
    a = a - a.mean(); b = b - b.mean()
    return abs(np.vdot(a, b))/np.sqrt(np.vdot(a, a).real*np.vdot(b, b).real)

print(f'H_c+ 重建：峰值位置 {pk}（点目标在 {(N//2, N//2)}），能量集中度 = {conc:.4f}')
print(f'sin 全息图单独重建：主像能量占比 = {ratio_sin:.3f}（另一半进入孪生像）')
print(f'|H_c- 重建| vs |2*z0 菲涅耳衍射|：相关系数 = {cc(np.abs(rec_cm), np.abs(fres2)):.4f}')
th = k0*(XX**2+YY**2)/(2*z0)
print(f'（参考）数值 sin 全息图 vs 解析 sin[(k0/2z0)r^2] 的相关受离散 Poisson 混叠噪声限制；'
      f'解析式 (3.5-4a/b) 本身由 i_Omegap 的实/虚部直接给出')

# ----- 6. 可视化（对应笔记图 sim-point-target.png） -----
fig, ax = plt.subplots(2, 3, figsize=(13.5, 8))
ext = [x[0]*1e3, x[-1]*1e3, x[0]*1e3, x[-1]*1e3]
ax[0,0].imshow(np.sin(th), extent=ext, cmap='gray')
ax[0,0].set_title('sin-FZP of a point (Eq. 3.5-4a, analytic)')
ax[0,1].imshow(np.cos(th), extent=ext, cmap='gray')
ax[0,1].set_title('cos-FZP of a point (Eq. 3.5-4b, analytic)')
ax[0,2].imshow(rcp**0.3, extent=ext, cmap='gray')
ax[0,2].set_title(f'H_c+ * h(z0): focused point (conc = {conc:.2f})')
ax[1,0].imshow(rsm**0.3, extent=ext, cmap='gray')
ax[1,0].set_title('sin-FZP alone: point + twin (50/50)')
ax[1,1].imshow(np.abs(rec_cm)**0.5, extent=ext, cmap='gray')
ax[1,1].set_title('H_c- * h(z0): defocused')
ax[1,2].imshow(np.abs(fres2)**0.5, extent=ext, cmap='gray')
ax[1,2].set_title('Fresnel diffraction @2z0 (match)')
for a in ax.flat:
    a.set_xlabel('x (mm)'); a.set_ylabel('y (mm)')
plt.tight_layout(); plt.show()


## tdfzp

`tdfzp.py`

In [ ]:
# ===== 书 3.6 节 · Table 3.2 TDFZP.m 的 Python 版：时间依赖菲涅耳波带片的运行条纹 =====
# TDFZP(x,y;z0,t) = A + B*sin[(k0/2*z0)(x^2+y^2) + Omega*t]     (3.6-1)
# 平面波（w0+Omega）与球面波（w0）干涉形成的"会跑的 FZP"：t 增大时同相位圆环半径缩小，
# 条纹向中心运动（running fringes toward the center）——OSH 就是用它光栅扫描 3-D 物体。
# 换瞳（p1=delta, p2=1）得 TDFZP(x,y;z0,-t)：条纹远离中心；sin 全息图不变、cos 全息图反号 (3.6-4)。
#
# 原书 TDFZP.m 参数：B=10.01e6（时间频率，任意）、D=6（空间尺度，任意）、t=linspace(0,1,35)，
# x,y=linspace(-2.5,2.5,256)，FZP = 1+sin(D*(x^2+y^2)+B*t)，逐帧归一化后 movie 播放。
# 本版固定 3 帧（t 使相位差可见）输出静态对比；想看动画可仿照原书逐帧绘制。
#
# MATLAB / Python 差异提示：
# 1) 原书三重循环 (ii,jj,kk) 用矩阵广播一行完成：D*(X^2+Y^2)+B*t。
# 2) 原书 image(256*FZP) 行方向向下；plt.imshow 加 origin='lower' 对齐数学坐标。
# 3) 改变条纹运动方向：把 sin 里的 +B*t 改成 -B*t（等价于 3.6-4 的 TDFZP(x,y;z0,-t)）。

import numpy as np
import matplotlib.pyplot as plt

Dc = 6.0                       # 空间尺度（原书 D=6，任意单位）
Bf = 10.01e6*2*np.pi*0.05      # 时间相位速度（原书 B=10.01e6，此处缩放使静态帧相位差可见）
xg = np.linspace(-2.5, 2.5, 256)
Xg, Yg = np.meshgrid(xg, xg)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for i, tt in enumerate([0.0, 0.35, 0.7]):
    FZP = 1 + np.sin(Dc*(Xg**2 + Yg**2) + Bf*tt)     # TDFZP (3.6-1)
    ax[i].imshow(FZP, cmap='gray', extent=[-2.5, 2.5, -2.5, 2.5], origin='lower')
    ax[i].set_title(f'TDFZP: sin[6r$^2$ + $\\Omega$t],  $\\Omega$t = {Bf*tt:.2f}')
    ax[i].axis('off')
plt.tight_layout(); plt.show()

# 条纹向中心运动的验证：跟踪相位 = D*r^2 + B*t = const 的半径 r(t) = sqrt((c - B*t)/D)，
# t 增大 -> r 减小 -> 圆环向中心收缩（书 3.6 节 "running fringes moving toward the center"）。
